# JXPaint Tutorial 3 — Speed, and varying cosmology with no interpolator rebuild

Two things that make JXPaint useful for inference:
1. it paints a full catalogue in ~0.2 s on the GPU (the production Julia XGPaint
   painter takes **108 s/catalogue**), and
2. **varying cosmology costs almost nothing** — the beam-convolved shape table is
   cosmology-independent, so it is built once and reused; only the geometry
   $\theta_{500}(M,z)$ is recomputed (~6 ms).

In [ ]:
import os, sys, time
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")  # share the GPU
sys.path.insert(0, os.path.join("..", "src"))   # run from tutorials/
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import jax
print("JAX devices:", jax.devices())

In [ ]:
import pandas as pd
from jxpaint.profiles.shape_table import load_beamed_table
from jxpaint.painting.gpu_native import paint_catalogue_gpu_native, compute_geometry
from jxpaint.cosmology import FlatLCDM
from jxpaint import constants as C

CAT = "/rds/rds-lxu/tsz_project/tsz_catalogue_benchmark/catalogue_bench_snr_0.csv"
st = load_beamed_table()
df = pd.read_csv(CAT)
z, M, lon, lat, y0 = (df.z.values, df.M.values, df.lon.values,
                      df.lat.values, df.y0_true.values)
XGPAINT_SECONDS = 108.07   # production paint_a10_y0true_2d_mpi.jl, self-reported

## 1. Speed vs XGPaint

In [ ]:
paint_catalogue_gpu_native(z, M, lon, lat, y0, st)         # warm up (compile)
ts = []
for _ in range(5):
    t0 = time.time(); m = paint_catalogue_gpu_native(z, M, lon, lat, y0, st)
    ts.append(time.time()-t0)
t = float(np.min(ts))
print(f"JXPaint (GPU):  {t:.3f} s / catalogue")
print(f"XGPaint (CPU):  {XGPAINT_SECONDS:.1f} s / catalogue")
print(f"speedup:        {XGPAINT_SECONDS/t:.0f}x   (bit-for-bit identical map)")

## 2. How does the paint time scale with $N_{\rm side}$?

For a fixed beam, each halo's paint radius $\theta_{\max}$ is fixed *in
steradians*, so the number of pixels it covers grows with the pixel density
$\propto N_{\rm side}^2$. The total number of painted pixel-contributions is
therefore $\sim N_{\rm halo}\,N_{\rm side}^2$, and the output map has
$12\,N_{\rm side}^2$ pixels. We thus expect **time $\propto N_{\rm side}^2$** at
high resolution, on top of a fixed per-call overhead floor that dominates at low
$N_{\rm side}$. (We use a 40k-halo subset so the highest $N_{\rm side}$ fits in
GPU memory; the scaling is independent of $N_{\rm halo}$.)

In [ ]:
sub = slice(0, 40000)
zz, MM, lo, la, yy = z[sub], M[sub], lon[sub], lat[sub], y0[sub]
nsides = np.array([128, 256, 512, 1024, 2048, 4096])
times = []
for nside in nsides:
    sc = nside/1024.0
    e_per, n_per = int(28*sc + 6), int(210*sc*sc + 40)     # caps scale with Nside
    kw = dict(nside=int(nside), chunk=20_000_000, e_per_halo=e_per, n_per_halo=n_per)
    paint_catalogue_gpu_native(zz, MM, lo, la, yy, st, **kw)        # compile
    tt = []
    for _ in range(3):
        t0 = time.time()
        paint_catalogue_gpu_native(zz, MM, lo, la, yy, st, **kw)
        tt.append(time.time() - t0)
    times.append(min(tt))
    print(f"nside={nside:5d}: {min(tt):.4f} s   ({12*nside**2/1e6:6.1f}M pixels)")
times = np.array(times)

In [ ]:
slope = np.polyfit(np.log(nsides[-2:]), np.log(times[-2:]), 1)[0]  # asymptotic (top two)
plt.figure(figsize=(7, 5))
plt.loglog(nsides, times, "o-", label="JXPaint warm paint")
ref = times[nsides == 2048][0] * (nsides/2048.0)**2
plt.loglog(nsides, ref, "k--", label=r"$\propto N_{\rm side}^2$")
plt.axhline(times[0], color="C2", ls=":", label="fixed overhead floor")
plt.xlabel(r"$N_{\rm side}$"); plt.ylabel("warm paint time [s]")
plt.title(f"Paint time vs $N_{{\\rm side}}$  (asymptotic slope = {slope:.2f})")
plt.legend(); plt.grid(alpha=.3, which="both"); plt.tight_layout(); plt.show()
print(f"asymptotic (2048->4096) log-log slope = {slope:.2f}  (Nside^2 -> 2.0)")

At low $N_{\rm side}$ the time is flat — dominated by fixed per-call costs (kernel
launch, geometry, disc setup). For $N_{\rm side}\gtrsim1024$ it follows the
$N_{\rm side}^2$ line, because both the number of painted pixel-contributions and
the map size grow as $N_{\rm side}^2$. (Total flux is conserved: the map sum
$\propto N_{\rm side}^2$ since the pixel area $\propto N_{\rm side}^{-2}$.)

## 3. Varying cosmology — the interpolator is **never rebuilt**

In XGPaint, changing cosmology means rebuilding the high-precision
(8192×8192, FFTLog-beamed) interpolator — expensive. In JXPaint the table is
cosmology-independent: we reuse the same `st` object for every cosmology and only
recompute the geometry. Below we paint the same catalogue under a grid of
cosmologies.

In [ ]:
cosmos = [FlatLCDM(h=h, Omega_m=Om) for h in (0.64, 0.6766, 0.70, 0.74)
          for Om in (0.27, 0.31, 0.35)]
paint_catalogue_gpu_native(z, M, lon, lat, y0, st, cosmo=cosmos[0])   # compile once

rows = []
for cz in cosmos:
    tg = []
    for _ in range(5):
        t0 = time.time()
        _, th500, _, _ = compute_geometry(z, M, lon, lat, y0, cz, C.B_BIAS)
        th500.block_until_ready(); tg.append(time.time()-t0)
    t0 = time.time(); mm = paint_catalogue_gpu_native(z, M, lon, lat, y0, st, cosmo=cz)
    tp = time.time()-t0
    rows.append((cz.h, cz.Omega_m, min(tg)*1e3, tp, mm.sum()))
res = pd.DataFrame(rows, columns=["h", "Omega_m", "geom_ms", "paint_s", "map_sum"])
print("interpolator rebuilds:", 0, " (same table object reused for all",
      len(cosmos), "cosmologies)")
res

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for h in sorted(res.h.unique()):
    sub = res[res.h == h]
    ax[0].plot(sub.Omega_m, sub.map_sum, "o-", label=f"h={h}")
ax[0].set_xlabel(r"$\Omega_m$"); ax[0].set_ylabel("map sum (total y)")
ax[0].set_title("Map varies with cosmology"); ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].bar(range(len(res)), res.geom_ms)
ax[1].set_xlabel("cosmology index"); ax[1].set_ylabel("geometry recompute [ms]")
ax[1].set_title("Per-cosmology cost = geometry only (~ms)"); ax[1].grid(alpha=.3, axis="y")
plt.tight_layout(); plt.show()

**Takeaway:** the beam-convolved interpolator is built/loaded **once** and reused
for every cosmology (0 rebuilds); the only cosmology-dependent work is the
vectorised geometry at ~6 ms/cosmology — negligible. This is what makes JXPaint
practical for cosmology inference where the painter is called thousands of times.